# VOSSA offline inference

Converts a whole source utterance to a target speaker in one pass, then measures how close the result is to each speaker.

For chunk-by-chunk conversion as in a live application, see `inference_streaming.ipynb`.

In [1]:
import sys
sys.path.append('..')
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from IPython import display as disp
from src.pretrained import load_vossa

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [2]:
# Downloads the checkpoints from the Hugging Face Hub on first use.
# To use files already on disk: load_vossa(local_dir="../checkpoints", device=device)
resynthesizer = load_vossa(device=device)

/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loss functions:  {'mel_fine_loss': 10.0, 'mel_coarse_loss': 2.0, 'adv_Gx_loss': 1.0, 'adv_Gx_feat_loss': 5.0, 'speaker_recon_loss': 30.0, 'speaker_vc_loss': 30.0, 'contrastive_loss': 10.0}


## Source and target audio

In [3]:
src_audio, _ = librosa.load("../samples/14_208_000001_000000.wav", sr=16000)
tgt_audio, _ = librosa.load("../samples/arctic_a0002.wav", sr=16000)
disp.display(disp.Audio(src_audio, rate=16000))
disp.display(disp.Audio(tgt_audio, rate=16000))

## Convert

The model takes `[1, 1, T]` waveforms at 16 kHz. The output is trimmed to the source length.

In [4]:
def to_tensor(audio):
    return torch.from_numpy(audio).view(1, 1, -1).to(device)

@torch.no_grad()
def convert(src_audio, tgt_audio):
    out = resynthesizer(to_tensor(src_audio), to_tensor(tgt_audio))
    return out.view(-1)[:len(src_audio)].cpu().numpy()

converted_audio = convert(src_audio, tgt_audio)
disp.Audio(converted_audio, rate=16000)

## Speaker similarity

Cosine similarities between global speaker embeddings, normalized by how similar the source and target already are: 0 means as similar as the source is to the target, 1 means identical. A good conversion has a high target similarity and a low source similarity.

In [5]:
@torch.no_grad()
def speaker_embedding(audio):
    return resynthesizer.get_speaker_embeddings(to_tensor(audio))[0].squeeze()

source_emb = speaker_embedding(src_audio)
target_emb = speaker_embedding(tgt_audio)
vc_emb = speaker_embedding(converted_audio)

sim_src_tgt = F.cosine_similarity(source_emb, target_emb, dim=0).item()
sim_syn_src = F.cosine_similarity(vc_emb, source_emb, dim=0).item()
sim_syn_tgt = F.cosine_similarity(vc_emb, target_emb, dim=0).item()

normalized_source_sim = (sim_syn_src - sim_src_tgt) / (1 - sim_src_tgt)
normalized_target_sim = (sim_syn_tgt - sim_src_tgt) / (1 - sim_src_tgt)

print(f"Similarity to source: {normalized_source_sim:.4f}")
print(f"Similarity to target: {normalized_target_sim:.4f}")

Similarity to source: -0.0549
Similarity to target: 0.9521
